# Phân tích dữ liệu Thương mại điện tử Olist (Brazil)

Dự án này thực hiện phân tích chuyên sâu trên bộ dữ liệu thương mại điện tử Olist của Brazil. Chúng ta sẽ sử dụng Python (`pandas`, `matplotlib`, `seaborn`) để phân tích các khía cạnh kinh doanh cốt lõi:

1. **Xu hướng doanh thu theo tháng & Doanh thu lũy kế**
2. **Phân khúc khách hàng (RFM)**
3. **Tác động của việc giao hàng trễ đối với điểm đánh giá của khách hàng (SLA & Reviews)**
4. **Phân tích doanh thu theo danh mục sản phẩm (Top 10)**
5. **Tỷ lệ giữ chân khách hàng (Cohort Analysis)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Thiết lập giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Nạp và Chuẩn bị Dữ liệu

In [ ]:
# Đường dẫn tới thư mục chứa dữ liệu
data_dir = '../data'

# Đọc các tệp dữ liệu
customers = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(data_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(data_dir, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(data_dir, 'olist_sellers_dataset.csv'))
category_translation = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))

print("Nạp dữ liệu thành công!")

In [ ]:
# Chuyển đổi các cột thời gian sang kiểu datetime
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

## 2. Bài toán 1: Xu hướng Doanh thu theo Tháng và Lũy kế

In [ ]:
# Tính tổng giá trị thanh toán cho mỗi đơn hàng
order_payments_agg = payments.groupby('order_id')['payment_value'].sum().reset_index(name='total_order_payment')

# Gộp với bảng orders và lọc các đơn hàng đã được giao thành công (delivered)
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
order_revenue = pd.merge(delivered_orders, order_payments_agg, on='order_id', how='inner')

# Trích xuất năm-tháng từ ngày mua hàng
order_revenue['month'] = order_revenue['order_purchase_timestamp'].dt.to_period('M')

# Tính doanh thu hàng tháng
monthly_rev = order_revenue.groupby('month')['total_order_payment'].sum().reset_index()
monthly_rev = monthly_rev.sort_values('month')
monthly_rev['cumulative_revenue'] = monthly_rev['total_order_payment'].cumsum()

# Chuyển month sang kiểu chuỗi để trực quan hóa dễ dàng hơn
monthly_rev['month_str'] = monthly_rev['month'].astype(str)
monthly_rev.head()

In [ ]:
# Trực quan hóa xu hướng doanh thu hàng tháng
fig, ax1 = plt.subplots(figsize=(14, 7))

# Vẽ biểu đồ cột doanh thu tháng
sns.barplot(data=monthly_rev, x='month_str', y='total_order_payment', ax=ax1, color='#3498db', alpha=0.8, label='Doanh thu tháng')
ax1.set_xlabel('Tháng', fontsize=12)
ax1.set_ylabel('Doanh thu (BRL)', fontsize=12, color='#2980b9')
ax1.tick_params(axis='y', labelcolor='#2980b9')
plt.xticks(rotation=45)

# Vẽ đường doanh thu lũy kế trên trục phụ
ax2 = ax1.twinx()
sns.lineplot(data=monthly_rev, x='month_str', y='cumulative_revenue', ax=ax2, color='#e74c3c', marker='o', linewidth=2.5, label='Lũy kế')
ax2.set_ylabel('Doanh thu lũy kế (BRL)', fontsize=12, color='#c0392b')
ax2.tick_params(axis='y', labelcolor='#c0392b')
ax2.grid(False)

plt.title('Xu hướng doanh thu theo tháng và lũy kế (Olist)', fontsize=16, fontweight='bold', pad=20)
fig.tight_layout()

# Đảm bảo thư mục images tồn tại và lưu biểu đồ
os.makedirs('../images', exist_ok=True)
plt.savefig('../images/monthly_revenue_trend.png', dpi=300)
plt.show()

## 3. Bài toán 2: Phân khúc Khách hàng (RFM Analysis)

In [ ]:
# Tính tổng giá trị thanh toán cho mỗi đơn hàng phục vụ tính Monetary
order_payments_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()

# Lọc các đơn hàng đã được giao thành công
delivered_orders = orders[orders['order_status'] == 'delivered']

# Gộp bảng khách hàng và đơn hàng để lấy customer_unique_id
customer_orders = pd.merge(customers, delivered_orders, on='customer_id', how='inner')
customer_orders = pd.merge(customer_orders, order_payments_agg, on='order_id', how='inner')

# Lấy mốc thời gian gần nhất trong cơ sở dữ liệu làm mốc tính Recency
max_date = orders['order_purchase_timestamp'].max()

# Tính toán RFM thô
rfm = customer_orders.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (max_date - x.max()).days, # Recency
    'order_id': 'nunique', # Frequency
    'payment_value': 'sum' # Monetary
}).reset_index()

rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']
rfm.head()

In [ ]:
# Phân phối của thuộc tính Monetary
plt.figure(figsize=(12, 6))
# Để trực quan hóa rõ hơn, chúng ta sẽ lọc các khách hàng có Monetary < 1000 BRL (vì phân phối bị lệch rất nhiều về bên phải)
sns.histplot(data=rfm[rfm['monetary'] < 1000], x='monetary', bins=50, kde=True, color='#2ecc71')
plt.title('Biểu đồ phân phối giá trị chi tiêu (Monetary) < 1000 BRL', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Giá trị chi tiêu (BRL)', fontsize=12)
plt.ylabel('Số lượng khách hàng', fontsize=12)
plt.savefig('../images/rfm_monetary_distribution.png', dpi=300)
plt.show()

## 4. Bài toán 3: Phân tích giao hàng trễ và ảnh hưởng đến điểm đánh giá

In [ ]:
# Lọc các đơn hàng đã được giao và có thông tin ngày nhận thực tế
delivered_df = orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].notna())].copy()

# Xác định trạng thái giao hàng
delivered_df['delivery_status'] = np.where(
    delivered_df['order_delivered_customer_date'] > delivered_df['order_estimated_delivery_date'], 
    'Late', 
    'On Time'
)

# Tính số ngày trễ (nếu giao sớm hoặc đúng hạn thì gán bằng 0)
delivered_df['days_late'] = (delivered_df['order_delivered_customer_date'] - delivered_df['order_estimated_delivery_date']).dt.days
delivered_df['days_late'] = delivered_df['days_late'].apply(lambda x: x if x > 0 else 0)

# Gộp với bảng đánh giá (reviews)
delivery_reviews = pd.merge(delivered_df, reviews, on='order_id', how='inner')

# Thống kê phân tích
delivery_summary = delivery_reviews.groupby('delivery_status').agg({
    'order_id': 'count',
    'review_score': 'mean',
    'days_late': 'mean'
}).reset_index()
delivery_summary.columns = ['delivery_status', 'total_orders', 'avg_review_score', 'avg_days_late']
delivery_summary

In [ ]:
# Trực quan hóa ảnh hưởng của giao hàng trễ đến điểm đánh giá trung bình
plt.figure(figsize=(10, 6))
sns.barplot(data=delivery_summary, x='delivery_status', y='avg_review_score', hue='delivery_status', palette=['#e74c3c', '#2ecc71'], legend=False)
plt.title('Điểm đánh giá trung bình theo Trạng thái Giao hàng (Late vs On Time)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Trạng thái Giao hàng', fontsize=12)
plt.ylabel('Điểm đánh giá trung bình (1-5)', fontsize=12)
plt.ylim(0, 5)

# Thêm giá trị số lên đầu các cột
for index, row in delivery_summary.iterrows():
    plt.text(index, row['avg_review_score'] + 0.15, f"{row['avg_review_score']:.2f}", ha='center', fontweight='bold', fontsize=12)

plt.savefig('../images/delivery_impact_reviews.png', dpi=300)
plt.show()

## 5. Bài toán 4: Phân tích danh mục sản phẩm (Top 10 Doanh thu)

In [ ]:
# Lọc các đơn hàng đã được giao thành công
delivered_orders = orders[orders['order_status'] == 'delivered']

# Gộp các bảng order_items, orders, products và category_translation
items_orders = pd.merge(order_items, delivered_orders, on='order_id', how='inner')
items_products = pd.merge(items_orders, products, on='product_id', how='inner')
items_categories = pd.merge(items_products, category_translation, on='product_category_name', how='left')

# Điền giá trị 'unknown' cho các danh mục trống
items_categories['product_category_name_english'] = items_categories['product_category_name_english'].fillna('unknown')

# Tính tổng doanh số (revenue) và số lượng bán (units_sold) theo từng danh mục
category_revenue = items_categories.groupby('product_category_name_english').agg({
    'price': 'sum',
    'order_id': 'count'
}).reset_index()
category_revenue.columns = ['category_name', 'total_revenue', 'units_sold']

# Lấy top 10 danh mục có doanh thu cao nhất
top_10_categories = category_revenue.sort_values('total_revenue', ascending=False).head(10)
top_10_categories

In [ ]:
# Trực quan hóa top 10 danh mục sản phẩm theo doanh thu
plt.figure(figsize=(14, 8))
sns.barplot(data=top_10_categories, x='total_revenue', y='category_name', hue='category_name', palette='viridis', legend=False)
plt.title('Top 10 danh mục sản phẩm mang lại Doanh thu lớn nhất', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Tổng Doanh thu (BRL)', fontsize=12)
plt.ylabel('Danh mục Sản phẩm (Tiếng Anh)', fontsize=12)
plt.savefig('../images/top_10_categories_revenue.png', dpi=300)
plt.show()

## 6. Bài toán 5: Tỷ lệ giữ chân khách hàng (Cohort Analysis)

In [ ]:
# Gộp bảng khách hàng và đơn hàng đã giao để thực hiện phân tích Cohort
cohort_data = pd.merge(customers, orders[orders['order_status'] == 'delivered'], on='customer_id', how='inner')

# Tạo cột tháng đặt hàng
cohort_data['order_month'] = cohort_data['order_purchase_timestamp'].dt.to_period('M')

# Tìm tháng đặt hàng đầu tiên của từng khách hàng (Cohort Month)
cohort_data['cohort_month'] = cohort_data.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min').dt.to_period('M')

# Nhóm dữ liệu theo cohort_month và order_month để đếm số khách hàng hoạt động
cohort_group = cohort_data.groupby(['cohort_month', 'order_month']).agg(retained_customers=('customer_unique_id', 'nunique')).reset_index()

# Tính khoảng cách tháng (month_index) giữa order_month và cohort_month
cohort_group['month_index'] = (cohort_group['order_month'] - cohort_group['cohort_month']).apply(lambda attr: attr.n)

# Biến đổi dữ liệu thành bảng chéo (Pivot table)
cohort_pivot = cohort_group.pivot(index='cohort_month', columns='month_index', values='retained_customers')

# Lấy kích thước ban đầu của từng Cohort (số khách hàng tại tháng 0)
cohort_sizes = cohort_pivot.iloc[:, 0]

# Chia tỷ lệ để tính Tỷ lệ giữ chân (Retention Rate %)
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

# Trực quan hóa Cohort Retention ở dạng Heatmap (giới hạn 12 tháng đầu để dễ quan sát)
plt.figure(figsize=(20, 10))
sns.heatmap(retention_matrix.iloc[:, 1:13], annot=True, fmt=".2f", cmap='Blues', cbar_kws={'label': 'Tỷ lệ giữ chân (%)'})
plt.title('Bản đồ giữ chân khách hàng - Cohort Retention Heatmap (%)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Chỉ số tháng (Tháng thứ n kể từ đơn hàng đầu tiên)', fontsize=12)
plt.ylabel('Nhóm tháng mua hàng đầu tiên (Cohort Month)', fontsize=12)
plt.show()